# Second-round Reviewer 1 — Experiment B recomputation under macro-F1 + per-fold baselines

Answers *Algorithms* Reviewer 1, round 2:

- **Comment 1**: recompute the prospective-feature ablation, the temporal validation and the country-based validation of Experiment B with **macro-F1** (positive-class F1 kept as a supplementary metric).
- **Comment 2**: report the **fold-specific majority-class baseline** for every country-based fold (and temporal test set) of Experiment A, next to the Random Forest F1-macro.

Same protocol as `review_1/expB_corrected_macro_f1_180926_respuesta.ipynb`: RF with `criterion='entropy'` (value found by the original search), RandomOverSampler inside the training data only, `average='macro'` passed explicitly. It also recomputes the pieces of the paper that still use positive-class F1 for Experiment B: the Decision Tree baseline (Table 15), the paired per-fold differences (Table 16) and the Wilcoxon tests.

Expected runtime: about 10 minutes. Experiment A models are **not** retrained (the RF F1-macro values per fold are already known); only its majority-class baselines are computed, which takes seconds.

At the end, the notebook saves `round2_results.json` (copy in `results/review_2/`). Results feed Tables 10 and 15–18 of the revised manuscript.

In [ ]:
# ── COLAB SETUP ─────────────────────────────────────────────────────────
# Step 1: mount Google Drive
from google.colab import drive
import os
# If already mounted, this line detects it automatically
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

# Step 2: install libraries not included in Colab
!pip install imbalanced-learn lime -q

# Step 3: copy the parquet file from Drive to the local Colab environment
# ─── ADJUST this path to the folder where you uploaded the parquet in your Drive ───
DRIVE_PARQUET = '/content/drive/MyDrive/fci-datos/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
LOCAL_PARQUET = '/content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
# ────────────────────────────────────────────────────────────────────────────

if not os.path.exists(LOCAL_PARQUET):
    import shutil
    shutil.copy(DRIVE_PARQUET, LOCAL_PARQUET)
    print(f'Parquet copied to {LOCAL_PARQUET}')
else:
    print(f'Parquet already available at {LOCAL_PARQUET}')

print('Setup complete.')

In [ ]:
import pandas as pd
import numpy as np
import time, json, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from imblearn.over_sampling import RandomOverSampler
from scipy.stats import wilcoxon

df_raw = pd.read_parquet(LOCAL_PARQUET)
for col in df_raw.select_dtypes(include='string').columns:
    df_raw[col] = df_raw[col].astype(object)
for col in ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME', 'YEAR']:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype(int)

df_raw['GROUP_CPV'] = df_raw['CPV'].astype(str).str[:2]
cat_cols = ['B_MULTIPLE_CAE', 'B_ON_BEHALF', 'CAE_TYPE', 'MAIN_ACTIVITY',
            'ISO_COUNTRY_CODE', 'TYPE_OF_CONTRACT', 'GROUP_CPV']
for col in cat_cols:
    df_raw[col] = df_raw[col].astype(str)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_arr = ohe.fit_transform(df_raw[cat_cols])
num_cols = ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS']
df_proc = pd.concat([df_raw[num_cols].reset_index(drop=True),
                     pd.DataFrame(ohe_arr, columns=ohe.get_feature_names_out(cat_cols))], axis=1)

df_raw['SME_WIN'] = df_raw['B_CONTRACTOR_SME'].astype(str).str.contains('y', case=False, na=False).astype(int)

used_features_B2 = [
    "B_MULTIPLE_CAE_n", "B_ON_BEHALF_n",
    "GROUP_CPV_45", "GROUP_CPV_33", "GROUP_CPV_15",
    "TYPE_OF_CONTRACT_w", "MAIN_ACTIVITY_health", "ISO_COUNTRY_CODE_si",
    "NUMBER_AWARDS", "LOTS_NUMBER", "NUMBER_OFFERS",
    "CAE_TYPE_3", "MAIN_ACTIVITY_general public\\services",
    "CAE_TYPE_4", "CAE_TYPE_5", "ISO_COUNTRY_CODE_lu"
]
used_features_B2_prosp = [f for f in used_features_B2 if f not in ('NUMBER_OFFERS', 'NUMBER_AWARDS')]

X_B2       = df_proc[used_features_B2].astype(float).to_numpy()
X_B2_prosp = df_proc[used_features_B2_prosp].astype(float).to_numpy()
y_B2  = df_raw['SME_WIN'].reset_index(drop=True).to_numpy()
y_A   = df_raw['MAIN_ACTIVITY'].astype(str).reset_index(drop=True).to_numpy()
years = df_raw['YEAR'].reset_index(drop=True).to_numpy()
paises = df_raw['ISO_COUNTRY_CODE'].reset_index(drop=True).to_numpy()

print('X_B2', X_B2.shape, '| X_B2_prosp', X_B2_prosp.shape, '| SME_WIN=1:', y_B2.mean().round(3))
assert X_B2.shape[1] == 16 and X_B2_prosp.shape[1] == 14

RESULTS = {}   # everything that goes into the paper ends up here

## Models (same hyperparameters as the paper; RF uses `entropy` for Experiment B)

In [ ]:
rf_B = RandomForestClassifier(criterion='entropy', max_features='sqrt', bootstrap=True,
                              max_samples=2/3, n_estimators=100, n_jobs=-1, random_state=42)
gb_B = GradientBoostingClassifier(max_features='sqrt', learning_rate=0.1,
                                  n_estimators=100, random_state=42)
dt_B = DecisionTreeClassifier(max_depth=5, random_state=42)

## Helper functions

In [ ]:
def binary_metrics(y_true, y_pred, y_score=None):
    out = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'f1_macro':  f1_score(y_true, y_pred, average='macro', zero_division=0),
        'pre_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'rec_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_pos':    f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None and len(np.unique(y_true)) == 2:
        out['auc'] = roc_auc_score(y_true, y_score)
    return out


def summarize(per_fold):
    keys = per_fold[0].keys()
    return {k: (float(np.mean([f[k] for f in per_fold])), float(np.std([f[k] for f in per_fold])))
            for k in keys}


def cv_binary(X, y, model, name, k=10):
    """Stratified k=10, RandomOverSampler inside the training fold, macro + positive-class metrics."""
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    ros = RandomOverSampler(random_state=42)
    per_fold = []
    t0 = time.time()
    for tr, te in skf.split(X, y):
        m = clone(model)
        Xb, yb = ros.fit_resample(X[tr], y[tr])
        m.fit(Xb, yb)
        per_fold.append(binary_metrics(y[te], m.predict(X[te]), m.predict_proba(X[te])[:, 1]))
    s = summarize(per_fold)
    print(f'{name:<45} macro-F1={s["f1_macro"][0]:.4f}±{s["f1_macro"][1]:.4f}  '
          f'pos-F1={s["f1_pos"][0]:.4f}±{s["f1_pos"][1]:.4f}  AUC={s["auc"][0]:.4f}±{s["auc"][1]:.4f}  '
          f'({(time.time()-t0)/60:.1f} min)')
    return {'per_fold': per_fold, 'summary': s}


def majority_predict(y_train, n):
    vals, counts = np.unique(y_train, return_counts=True)
    return np.full(n, vals[np.argmax(counts)])

## 1. Stratified k=10 CV — Experiment B (full features): RF, GB, Decision Tree
RF and GB should reproduce 0.5863 / 0.5569. The Decision Tree value replaces the earlier 0.6753 (positive-class F1) in Table 15.

In [ ]:
cv_rf = cv_binary(X_B2, y_B2, rf_B, 'RF (entropy), 16 features')
cv_gb = cv_binary(X_B2, y_B2, gb_B, 'GB, 16 features')
cv_dt = cv_binary(X_B2, y_B2, dt_B, 'Decision Tree (depth=5), 16 features')

assert abs(cv_rf['summary']['f1_macro'][0] - 0.5863) < 0.001, 'RF does not reproduce 0.5863 — check setup'
RESULTS['cv_B'] = {'rf': cv_rf['summary'], 'gb': cv_gb['summary'], 'dt': cv_dt['summary']}

## 2. Paired per-fold differences and Wilcoxon (Table 16 rows for Experiment B, Table 15 caption)

In [ ]:
def paired(a, b, label):
    fa = np.array([f['f1_macro'] for f in a['per_fold']])
    fb = np.array([f['f1_macro'] for f in b['per_fold']])
    d = fa - fb
    w1 = wilcoxon(fa, fb, alternative='greater')
    w2 = wilcoxon(fa, fb, alternative='two-sided')
    res = {'mean': float(d.mean()), 'std': float(d.std()), 'min': float(d.min()), 'max': float(d.max()),
           'wins': int((d > 0).sum()), 'p_one_sided': float(w1.pvalue), 'p_two_sided': float(w2.pvalue)}
    print(f'{label:<18} ΔF1-macro = {res["mean"]:+.4f} ± {res["std"]:.4f}  range [{res["min"]:.3f}, {res["max"]:.3f}]  '
          f'wins {res["wins"]}/10  p1={res["p_one_sided"]:.4f}  p2={res["p_two_sided"]:.4f}')
    return res

RESULTS['paired_B'] = {'rf_minus_gb': paired(cv_rf, cv_gb, 'RF − GB (B)'),
                       'rf_minus_dt': paired(cv_rf, cv_dt, 'RF − DT (B)')}

## 3. Prospective-feature ablation (Table 10) — Comment 1

In [ ]:
cv_rf_prosp = cv_binary(X_B2_prosp, y_B2, rf_B, 'RF (entropy), 14 prospective features')
RESULTS['ablation_B'] = {'full': cv_rf['summary'], 'prospective': cv_rf_prosp['summary']}

for name, s in [('Full (16)', cv_rf['summary']), ('Prospective (14)', cv_rf_prosp['summary'])]:
    print(f'{name:<18} & ${s["f1_macro"][0]:.4f} \\pm {s["f1_macro"][1]:.4f}$ & '
          f'${s["f1_pos"][0]:.4f} \\pm {s["f1_pos"][1]:.4f}$ & ${s["auc"][0]:.4f} \\pm {s["auc"][1]:.4f}$ \\\\')

## 4. Temporal validation (train 2015–2021, test 2022 / 2023 / combined)
Experiment B under macro-F1 (Comment 1) and majority-class baselines for both experiments (Comment 2).

In [ ]:
ros = RandomOverSampler(random_state=42)
train = years <= 2021
tests = {'2022': years == 2022, '2023': years == 2023, '2022-2023': np.isin(years, [2022, 2023])}

m = clone(rf_B)
Xb, yb = ros.fit_resample(X_B2[train], y_B2[train])
m.fit(Xb, yb)

RESULTS['temporal'] = {}
for label, mask in tests.items():
    rB = binary_metrics(y_B2[mask], m.predict(X_B2[mask]), m.predict_proba(X_B2[mask])[:, 1])
    baseB = f1_score(y_B2[mask], majority_predict(y_B2[train], mask.sum()), average='macro', zero_division=0)
    baseA = f1_score(y_A[mask], majority_predict(y_A[train], mask.sum()), average='macro', zero_division=0)
    RESULTS['temporal'][label] = {'B': rB, 'B_majority_f1_macro': baseB, 'A_majority_f1_macro': baseA,
                                  'n': int(mask.sum())}
    print(f'test {label:<10} n={mask.sum():>6,}  B: macro-F1={rB["f1_macro"]:.4f} pos-F1={rB["f1_pos"]:.4f} '
          f'AUC={rB["auc"]:.4f} | B majority macro-F1={baseB:.4f} | A majority macro-F1={baseA:.4f}')

## 5. Country-based validation (GroupKFold on ISO_COUNTRY_CODE, k=5)
Experiment B under macro-F1 (Comment 1) and fold-specific majority-class baselines for both experiments (Comment 2).

The Experiment A Random Forest F1-macro per fold is taken from `review_1/PROCESO_MODELADO_CV_EN_perclass_100926-respuesta.ipynb`, Step 7d (same folds: GroupKFold is deterministic and the held-out countries match). The check below asserts that the folds are the same.

In [ ]:
RF_A_COUNTRY_F1 = [0.1569, 0.1469, 0.1389, 0.1239, 0.1550]   # from Step 7d of PROCESO_MODELADO_CV_EN_perclass_100926-respuesta.ipynb
EXPECTED_FOLDS = [['pl'], ['at','cz','lu','ro','sk'],
                  ['be','cy','ee','fi','fr','hr','mk','mt','nl','se'],
                  ['bg','gr','hu','it','lt','no','pt'], ['de','dk','es','ie','lv','si','uk']]

gkf = GroupKFold(n_splits=5)
per_fold = []
for i, (tr, te) in enumerate(gkf.split(X_B2, y_B2, groups=paises)):
    held = sorted(set(paises[te]))
    assert held == EXPECTED_FOLDS[i], f'Fold {i+1} differs from the main notebook: {held}'

    mB = clone(rf_B)
    Xb, yb = ros.fit_resample(X_B2[tr], y_B2[tr])
    mB.fit(Xb, yb)
    rB = binary_metrics(y_B2[te], mB.predict(X_B2[te]), mB.predict_proba(X_B2[te])[:, 1])

    baseB = f1_score(y_B2[te], majority_predict(y_B2[tr], len(te)), average='macro', zero_division=0)
    yA_maj = majority_predict(y_A[tr], len(te))
    baseA = f1_score(y_A[te], yA_maj, average='macro', zero_division=0)
    n_classes_A = len(np.unique(y_A[te]))

    row = {'fold': i + 1, 'countries': held, 'n_test': int(len(te)),
           'A_rf_f1_macro': RF_A_COUNTRY_F1[i], 'A_majority_f1_macro': baseA,
           'A_majority_class': str(yA_maj[0]), 'A_classes_in_test': n_classes_A,
           'B': rB, 'B_majority_f1_macro': baseB}
    per_fold.append(row)
    print(f'Fold {i+1} {",".join(held):<32} n={len(te):>6,} | A: RF={RF_A_COUNTRY_F1[i]:.4f} '
          f'majority={baseA:.4f} ({n_classes_A} classes) | B: macro-F1={rB["f1_macro"]:.4f} '
          f'pos-F1={rB["f1_pos"]:.4f} AUC={rB["auc"]:.4f} majority={baseB:.4f}')

RESULTS['country'] = {'per_fold': per_fold,
                      'B_summary': summarize([r['B'] for r in per_fold]),
                      'A_rf_mean_std': (float(np.mean(RF_A_COUNTRY_F1)), float(np.std(RF_A_COUNTRY_F1))),
                      'A_folds_below_baseline': int(sum(r['A_rf_f1_macro'] <= r['A_majority_f1_macro'] for r in per_fold))}
s = RESULTS['country']['B_summary']
print(f'nB country mean: macro-F1={s["f1_macro"][0]:.4f}±{s["f1_macro"][1]:.4f}  '
      f'pos-F1={s["f1_pos"][0]:.4f}±{s["f1_pos"][1]:.4f}')
print('Experiment A folds at or below their own majority baseline:', RESULTS['country']['A_folds_below_baseline'])

## 6. Save everything

In [ ]:
def to_py(o):
    if isinstance(o, dict):  return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)):  return int(o)
    return o

with open('round2_results.json', 'w') as fh:
    json.dump(to_py(RESULTS), fh, indent=2)
print('Saved round2_results.json')

from google.colab import files
files.download('round2_results.json')